# **SUPPORT**
## <span style="color:red">***Low-Power Test***</span>

### ***Initialization***

In [ ]:
"""Change to the location of the SUPPORT repo"""
path_support = "/Users/garfinkeljb/Downloads/s/SUPPORT" # NO ENDING /

In [ ]:
import sys
from pathlib import Path
import os
os.chdir(f'{path_support}/notebooks/')
sys.path.insert(0, str(Path('./').resolve().parent.parent))
import multiprocessing
multiprocessing.freeze_support()

### ***Basic configuration***
``` To modify parameters, go to notebooks/parameters.json ```

In [ ]:
dir_noisy = "/Volumes/NVRserverStoragePool2/Josh_Garfinkel/01_projects/01_dinamiCa/20260806_validationSet/01_data/01_raw" # input directory
dimensionality = "3+1D" # options: "2+1D" (planar + time), "3D" (static volumetric), or "3+1D" (volumetric + time)
power_percent = [100, 90, 80, 70, 60, 50, 40]

# Photon scale. None estimates it once per raw/original-SUPPORT pair.
# Use a positive float here to bypass estimation with a fixed override.
photons_per_unit = None
# Keep only a measured laser-independent additive detector baseline fixed.
# Low electronic-noise variance does not imply that this offset is zero.
# Optical and out-of-focus background above this level scales with power.
photon_zero_level = 0.0
# The fit-derived pedestal is extrapolated; prefer a dark/laser-off value.
use_estimated_photon_pedestal = False
photon_estimator_frame_start = 60
photon_estimator_frame_end_margin = 60
photon_estimator_max_pairs = 50
photon_estimator_sample_count = 40_000
photon_estimator_random_seed = 7
allow_provisional_photon_estimate = False

# node-dependent: change if deviating from typical node specs
n_cpu = 8
gpu = '0'

rm_tmp = False # if True, delete simulated power TIFFs; keep denoised originals
rm_dir = False # if True, store training artifacts under notebooks/tmp
checkpoints = False # if True, save resumable checkpoints (four times per epoch when train_gen is True)
resume = True # if True, continue from the latest complete checkpoint when available
rm_intermediaries = True # if True, keep only the most recently saved checkpoint

paths_noisy = sorted(
    path for path in Path(dir_noisy).rglob('*')
    if path.is_file() and path.suffix.casefold() in {'.tif', '.tiff'}
)
for p in paths_noisy:
    print(p.parent.stem)

### ***Staged original-image and power sweep***

With `photons_per_unit = None`, the raw/SUPPORT photon-transfer fit is recomputed independently for every discovered raw TIFF after its original image is denoised. Its SUPPORT-unit result is then reused only for power levels from that source. Calibration excludes the first and final 60 frames by default; at most 50 adjacent pairs are spread across that central window. The simulator uses the two-photon relationship $I\propto P^2$ and scales all photon-generating intensity, including out-of-focus fluorescence. Only `photon_zero_level` remains fixed; this is an additive detector baseline, not optical background subtraction. Set it from dark/laser-off frames when possible. `use_estimated_photon_pedestal = True` explicitly opts into the fit-derived, model-dependent pedestal instead (and cannot be combined with a fixed photon scale or nonzero `photon_zero_level`). Independent Poisson sampling matches marginal shot-noise variance but does not reproduce measured scanner-correlated noise.

In [ ]:
from SUPPORT.notebooks import snr_test as staged_snr

repository = Path(path_support).expanduser().resolve()
output_root = repository / 'notebooks' / 'outputs'
original_tasks = staged_snr.build_original_tasks(paths_noisy, output_root)
power_tasks = staged_snr.build_sweep_tasks(paths_noisy, power_percent, output_root)
configuration = staged_snr.SweepConfiguration(
    repository=repository,
    input_root=Path(dir_noisy).expanduser().resolve(),
    parameters_path=repository / 'notebooks' / 'parameters.json',
    output_root=output_root,
    dimensionality=dimensionality,
    n_cpu=n_cpu,
    rm_tmp=rm_tmp,
    rm_dir=rm_dir,
    checkpoints=checkpoints,
    resume=resume,
    rm_intermediaries=rm_intermediaries,
    photons_per_unit=photons_per_unit,
    photon_zero_level=photon_zero_level,
    use_estimated_photon_pedestal=use_estimated_photon_pedestal,
    photon_estimator_frame_start=photon_estimator_frame_start,
    photon_estimator_frame_end_margin=photon_estimator_frame_end_margin,
    photon_estimator_max_pairs=photon_estimator_max_pairs,
    photon_estimator_sample_count=photon_estimator_sample_count,
    photon_estimator_random_seed=photon_estimator_random_seed,
    allow_provisional_photon_estimate=allow_provisional_photon_estimate,
)
selected_gpus = staged_snr.select_gpu_slots(
    [int(gpu)], staged_snr._discover_visible_devices()
)
results = staged_snr.run_staged_sweep(
    original_tasks, power_tasks, selected_gpus, configuration
)
failures = [result for result in results if not result.success]
if failures:
    raise RuntimeError(f'{len(failures)} SNR power job(s) failed')
